# 1. Carga de datasets

Se cargan los tres datasets previamente preparados y normalizados que servirán como base para la construcción del dataset maestro.

In [3]:
import pandas as pd
import numpy as np

results = pd.read_csv(
    "data/results1_normalized.csv"
)

fifa = pd.read_csv(
    "data/fifa_normalized.csv"
)

elo = pd.read_csv(
    "data/elo_normalized.csv"
)

# 2. Conversión de variables temporales

Se convierten las variables de fecha al formato datetime para permitir búsquedas temporales y posteriores uniones entre datasets.

In [4]:
results["date"] = pd.to_datetime(
    results["date"]
)

fifa["rank_date"] = pd.to_datetime(
    fifa["rank_date"]
)

elo["date"] = pd.to_datetime(
    elo["date"]
)

# 3. Verificación de los datasets cargados

In [5]:
results.shape

(22978, 8)

In [9]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,goal_diff
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,False,1.0
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,False,7.0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,False,0.0
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,False,0.0
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,True,0.0


In [6]:
fifa.shape

(57203, 4)

In [7]:
elo.shape

(5073, 3)

# 4. Creación de la importancia del torneo

No todos los partidos tienen la misma relevancia competitiva.

Se crea una variable denominada `tournament_weight` para representar la importancia relativa de cada competición.

Los Mundiales reciben la máxima valoración, seguidos por competiciones continentales, fases de clasificación, competiciones secundarias y amistosos.

In [12]:
results["tournament"].value_counts()

tournament
Friendly                                7807
FIFA World Cup qualification            5677
UEFA Euro qualification                 1532
African Cup of Nations qualification    1364
UEFA Nations League                      658
                                        ... 
East Asian Games                           1
Corsica Cup                                1
Copa Confraternidad                        1
CONMEBOL–UEFA Cup of Champions             1
South Asian Super Cup                      1
Name: count, Length: 98, dtype: int64

In [13]:
sorted(
    results["tournament"].unique()
)

['ABCS Tournament',
 'AFC Asian Cup',
 'AFC Asian Cup qualification',
 'AFC Challenge Cup',
 'AFC Challenge Cup qualification',
 'AFF Championship',
 'AFF Championship qualification',
 'ASEAN Championship',
 'ASEAN Championship qualification',
 'African Cup of Nations',
 'African Cup of Nations qualification',
 'Afro-Asian Games',
 'Al Ain International Cup',
 'Amílcar Cabral Cup',
 'Arab Cup',
 'Arab Cup qualification',
 'Baltic Cup',
 'CAFA Nations Cup',
 'CECAFA Cup',
 'CFU Caribbean Cup',
 'CFU Caribbean Cup qualification',
 'CONCACAF Nations League',
 'CONCACAF Nations League qualification',
 'CONCACAF Series',
 'CONMEBOL–UEFA Cup of Champions',
 'COSAFA Cup',
 'COSAFA Cup qualification',
 'Canadian Shield',
 'Confederations Cup',
 'Copa América',
 'Copa América qualification',
 'Copa Confraternidad',
 'Copa Paz del Chaco',
 'Copa del Pacífico',
 'Corsica Cup',
 "Coupe de l'Outre-Mer",
 'Cup of Ancient Civilizations',
 'Cyprus International Tournament',
 'Dragon Cup',
 'EAFF Champ

In [14]:
def assign_tournament_weight(tournament):

    if tournament == "FIFA World Cup":
        return 10

    elif tournament in [
        "UEFA Euro",
        "Copa América",
        "African Cup of Nations",
        "AFC Asian Cup",
        "CONCACAF Gold Cup",
        "Confederations Cup"
    ]:
        return 8

    elif "qualification" in tournament.lower():
        return 6

    elif "nations league" in tournament.lower():
        return 4

    elif tournament == "Friendly":
        return 1

    else:
        return 3

In [15]:
results["tournament_weight"] = (
    results["tournament"]
    .apply(assign_tournament_weight)
)

In [16]:
results[
    ["tournament", "tournament_weight"]
].drop_duplicates().sort_values(
    "tournament_weight",
    ascending=False
)

,tournament,tournament_weight
2242,FIFA World Cup,10
20,African Cup of Nations,8
514,UEFA Euro,8
1456,Confederations Cup,8
1611,Copa América,8
...,...,...
22051,Mapinduzi Cup,3
22757,CONCACAF Series,3
22496,South Asian Super Cup,3
22779,Al Ain International Cup,3


In [17]:
results["tournament_weight"].value_counts()

tournament_weight
6     9517
1     7807
3     2988
8     1378
4      904
10     384
Name: count, dtype: int64

In [18]:
results["neutral"].value_counts()

neutral
False    16878
True      6100
Name: count, dtype: int64

# 5. Selección de variables relevantes

Se conservan únicamente las variables necesarias para la construcción del dataset maestro.

In [8]:
results.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'goal_diff'],
      dtype='str')

In [ ]:
results = results[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "tournament_weight",
        "neutral",
        "goal_diff"
    ]
]

In [ ]:
fifa.columns

In [ ]:
fifa = fifa[
    [
        "rank_date",
        "country_full",
        "rank",
        "total_points"
    ]
]

In [ ]:
elo.columns

In [ ]:
elo = elo[
    [
        "date",
        "team",
        "rating"
    ]
]

# 6. Construcción de la variable objetivo

Se verifica la variable objetivo del problema.

La diferencia de goles representa el margen de victoria o derrota y será utilizada posteriormente como variable a predecir por los modelos de Machine Learning.

In [19]:
results["goal_diff"].describe()

count    22978.000000
mean         0.526155
std          2.229107
min        -21.000000
25%         -1.000000
50%          0.000000
75%          2.000000
max         31.000000
Name: goal_diff, dtype: float64

In [20]:
results["goal_diff"].value_counts().sort_index()

goal_diff
-21.0       1
-15.0       2
-14.0       1
-13.0       1
-12.0       1
-11.0       2
-10.0      11
-9.0        4
-8.0       23
-7.0       40
-6.0       89
-5.0      141
-4.0      330
-3.0      789
-2.0     1700
-1.0     3345
 0.0     5444
 1.0     4793
 2.0     2941
 3.0     1586
 4.0      824
 5.0      428
 6.0      212
 7.0      123
 8.0       58
 9.0       29
 10.0      20
 11.0      13
 12.0       7
 13.0       4
 14.0       5
 15.0       2
 16.0       2
 17.0       2
 19.0       2
 20.0       1
 22.0       1
 31.0       1
Name: count, dtype: int64

# 7. Integración temporal del Ranking FIFA y Elo

Se incorporan al dataset principal las variables de Ranking FIFA y Elo vigentes en la fecha de cada partido.

Para evitar utilizar información futura, se emplea un merge temporal que asigna a cada encuentro el último ranking disponible anterior a la fecha del partido.

In [21]:
results["date"] = pd.to_datetime(results["date"])
fifa["rank_date"] = pd.to_datetime(fifa["rank_date"])
elo["date"] = pd.to_datetime(elo["date"])

In [22]:
results = results.sort_values("date")

fifa = fifa.sort_values("rank_date")

elo = elo.sort_values("date")

In [23]:
mundial_final = results.copy()

In [24]:
print(results.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'goal_diff', 'tournament_weight']


In [25]:
print(fifa.columns.tolist())

['rank_date', 'country_full', 'rank', 'total_points']


In [26]:
print(elo.columns.tolist())

['date', 'team', 'rating']


## 7.1 Preparación del Ranking FIFA

Se preparan los datos del Ranking FIFA para asociar a cada partido el ranking y puntuación vigentes en la fecha del encuentro.

In [27]:
# Creación FIFA Home

fifa_home = fifa.rename(
    columns={
        "country_full": "home_team",
        "rank": "fifa_rank_home",
        "total_points": "fifa_points_home"
    }
)

In [28]:
# Creación FIFA Away

fifa_away = fifa.rename(
    columns={
        "country_full": "away_team",
        "rank": "fifa_rank_away",
        "total_points": "fifa_points_away"
    }
)

In [29]:
fifa_home.head()

,rank_date,home_team,fifa_rank_home,fifa_points_home
0,2000-01-19,Gabon,72.0,456.0
128,2000-01-19,Guam,200.0,4.0
129,2000-01-19,Mongolia,199.0,6.0
130,2000-01-19,American Samoa,198.0,6.0
131,2000-01-19,Somalia,197.0,14.0


In [30]:
fifa_away.head()

,rank_date,away_team,fifa_rank_away,fifa_points_away
0,2000-01-19,Gabon,72.0,456.0
128,2000-01-19,Guam,200.0,4.0
129,2000-01-19,Mongolia,199.0,6.0
130,2000-01-19,American Samoa,198.0,6.0
131,2000-01-19,Somalia,197.0,14.0


## 7.2 Integración del Ranking FIFA del equipo local

In [31]:
mundial_final = pd.merge_asof(
    mundial_final.sort_values("date"),
    fifa_home.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="home_team",
    direction="backward"
)

In [32]:
mundial_final.shape

(22978, 12)

In [33]:
mundial_final[
    [
        "date",
        "home_team",
        "fifa_rank_home",
        "fifa_points_home"
    ]
].head()

,date,home_team,fifa_rank_home,fifa_points_home
0,2000-01-04,Egypt,NaN,NaN
1,2000-01-07,Tunisia,NaN,NaN
2,2000-01-08,Trinidad and Tobago,NaN,NaN
3,2000-01-09,Burkina Faso,NaN,NaN
4,2000-01-09,Guatemala,NaN,NaN


## 7.3 Integración del Ranking FIFA del equipo visitante

In [34]:
mundial_final = pd.merge_asof(
    mundial_final.sort_values("date"),
    fifa_away.sort_values("rank_date"),
    left_on="date",
    right_on="rank_date",
    by="away_team",
    direction="backward"
)

In [35]:
mundial_final.shape

(22978, 15)

In [36]:
mundial_final[
    [
        "date",
        "away_team",
        "fifa_rank_away",
        "fifa_points_away"
    ]
].head()

,date,away_team,fifa_rank_away,fifa_points_away
0,2000-01-04,Togo,NaN,NaN
1,2000-01-07,Togo,NaN,NaN
2,2000-01-08,Canada,NaN,NaN
3,2000-01-09,Gabon,NaN,NaN
4,2000-01-09,Armenia,NaN,NaN


In [37]:
mundial_final.shape

(22978, 15)

In [38]:
mundial_final.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,goal_diff,tournament_weight,rank_date_x,fifa_rank_home,fifa_points_home,rank_date_y,fifa_rank_away,fifa_points_away
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,False,1.0,1,NaT,NaN,NaN,NaT,NaN,NaN
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,False,7.0,1,NaT,NaN,NaN,NaT,NaN,NaN
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,False,0.0,1,NaT,NaN,NaN,NaT,NaN,NaN
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,False,0.0,1,NaT,NaN,NaN,NaT,NaN,NaN
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,True,0.0,1,NaT,NaN,NaN,NaT,NaN,NaN


In [39]:
mundial_final[
    mundial_final["fifa_rank_home"].notna()
].head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,goal_diff,tournament_weight,rank_date_x,fifa_rank_home,fifa_points_home,rank_date_y,fifa_rank_away,fifa_points_away
15,2000-01-19,Panama,Guatemala,2.0,0.0,Friendly,False,2.0,1,2000-01-19,139.0,214.0,2000-01-19,73.0,454.0
16,2000-01-19,Togo,Egypt,1.0,0.0,Friendly,False,1.0,1,2000-01-19,87.0,413.0,2000-01-19,38.0,558.0
17,2000-01-20,Burkina Faso,Algeria,1.0,0.0,Friendly,False,1.0,1,2000-01-19,74.0,453.0,2000-01-19,86.0,421.0
18,2000-01-20,Malta,Qatar,2.0,0.0,Friendly,False,2.0,1,2000-01-19,116.0,308.0,2000-01-19,107.0,357.0
19,2000-01-21,New Zealand,South Korea,0.0,1.0,Friendly,False,-1.0,1,2000-01-19,100.0,381.0,NaT,NaN,NaN


In [40]:
mundial_final["fifa_rank_home"].notna().sum()

np.int64(20905)

In [41]:
mundial_final["fifa_rank_away"].notna().sum()

np.int64(21048)

In [42]:
mundial_final = mundial_final.drop(
    columns=[
        "rank_date_x",
        "rank_date_y"
    ]
)

In [43]:
mundial_final.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'goal_diff', 'tournament_weight',
       'fifa_rank_home', 'fifa_points_home', 'fifa_rank_away',
       'fifa_points_away'],
      dtype='str')

# 8. Integración temporal del rating Elo

Se incorpora el rating Elo vigente para cada selección en la fecha de cada encuentro.

Al igual que con el Ranking FIFA, únicamente se utiliza información disponible antes de la celebración del partido.

In [44]:
elo_home = elo.rename(
    columns={
        "team": "home_team",
        "rating": "elo_home"
    }
)

In [45]:
elo_away = elo.rename(
    columns={
        "team": "away_team",
        "rating": "elo_away"
    }
)

In [46]:
mundial_final = pd.merge_asof(
    mundial_final.sort_values("date"),
    elo_home.sort_values("date"),
    on="date",
    by="home_team",
    direction="backward"
)

In [47]:
mundial_final.shape

(22978, 14)

In [48]:
mundial_final = pd.merge_asof(
    mundial_final.sort_values("date"),
    elo_away.sort_values("date"),
    on="date",
    by="away_team",
    direction="backward"
)

In [49]:
mundial_final.shape

(22978, 15)

In [50]:
mundial_final[
    [
        "home_team",
        "elo_home",
        "away_team",
        "elo_away"
    ]
].head()

,home_team,elo_home,away_team,elo_away
0,Egypt,NaN,Togo,NaN
1,Tunisia,NaN,Togo,NaN
2,Trinidad and Tobago,NaN,Canada,NaN
3,Burkina Faso,NaN,Gabon,NaN
4,Guatemala,NaN,Armenia,NaN


In [51]:
mundial_final["elo_home"].notna().sum()

np.int64(20900)

In [52]:
mundial_final["elo_away"].notna().sum()

np.int64(20795)

In [53]:
mundial_final[
    [
        "fifa_rank_home",
        "fifa_rank_away",
        "elo_home",
        "elo_away"
    ]
].notna().all(axis=1).sum()

np.int64(16714)

In [54]:
mundial_final.isna().sum()

date                    0
home_team               0
away_team               0
home_score              0
away_score              0
tournament              0
neutral                 0
goal_diff               0
tournament_weight       0
fifa_rank_home       2073
fifa_points_home     2073
fifa_rank_away       1930
fifa_points_away     1930
elo_home             2078
elo_away             2183
dtype: int64

In [57]:
mundial_final[
    mundial_final["fifa_rank_home"].isna()
][
    ["date","home_team","away_team"]
].head(20)

,date,home_team,away_team
0,2000-01-04,Egypt,Togo
1,2000-01-07,Tunisia,Togo
2,2000-01-08,Trinidad and Tobago,Canada
3,2000-01-09,Burkina Faso,Gabon
4,2000-01-09,Guatemala,Armenia
5,2000-01-09,Ivory Coast,Egypt
6,2000-01-09,Mexico,Iran
7,2000-01-11,Bermuda,Canada
8,2000-01-11,Burkina Faso,Cameroon
9,2000-01-13,Senegal,Cameroon


In [58]:
mundial_final[
    mundial_final["elo_home"].isna()
][
    ["date","home_team","away_team"]
].head(20)

,date,home_team,away_team
0,2000-01-04,Egypt,Togo
1,2000-01-07,Tunisia,Togo
2,2000-01-08,Trinidad and Tobago,Canada
3,2000-01-09,Burkina Faso,Gabon
4,2000-01-09,Guatemala,Armenia
5,2000-01-09,Ivory Coast,Egypt
6,2000-01-09,Mexico,Iran
7,2000-01-11,Bermuda,Canada
8,2000-01-11,Burkina Faso,Cameroon
9,2000-01-13,Senegal,Cameroon


## 8. Variables derivadas FIFA

Se generan variables que representan la diferencia de nivel entre ambas selecciones según el ranking FIFA.

In [55]:
# Diferencias FIFA

mundial_final["fifa_rank_diff"] = (
    mundial_final["fifa_rank_away"]
    - mundial_final["fifa_rank_home"]
)

mundial_final["fifa_points_diff"] = (
    mundial_final["fifa_points_home"]
    - mundial_final["fifa_points_away"]
)

## 9. Variables derivadas Elo

Se calcula la diferencia de rating Elo entre ambas selecciones.

In [ ]:
# Diferencias Elo

mundial_final["elo_diff"] = (
    mundial_final["elo_home"]
    - mundial_final["elo_away"]
)

In [60]:
mundial_final[
    [
        "home_team",
        "away_team",
        "fifa_rank_diff",
        "fifa_points_diff",
        "elo_diff"
    ]
].head(20)

,home_team,away_team,fifa_rank_diff,fifa_points_diff,elo_diff
0,Egypt,Togo,NaN,NaN,NaN
1,Tunisia,Togo,NaN,NaN,NaN
2,Trinidad and Tobago,Canada,NaN,NaN,NaN
3,Burkina Faso,Gabon,NaN,NaN,NaN
4,Guatemala,Armenia,NaN,NaN,NaN
5,Ivory Coast,Egypt,NaN,NaN,NaN
6,Mexico,Iran,NaN,NaN,NaN
7,Bermuda,Canada,NaN,NaN,NaN
8,Burkina Faso,Cameroon,NaN,NaN,NaN
9,Senegal,Cameroon,NaN,NaN,NaN


## 10. Variables de rendimiento reciente

Se generan variables basadas en los últimos partidos disputados por cada selección antes de cada encuentro.

Estas variables permiten representar el estado de forma reciente de los equipos.

In [61]:
home_df = mundial_final[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_df.columns = [
    "date",
    "team",
    "opponent",
    "goals_for",
    "goals_against"
]

In [62]:
away_df = mundial_final[
    [
        "date",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_df.columns = [
    "date",
    "team",
    "opponent",
    "goals_for",
    "goals_against"
]

In [63]:
matches_long = pd.concat(
    [home_df, away_df],
    ignore_index=True
)

In [65]:
matches_long.head(20)

,date,team,opponent,goals_for,goals_against
0,2000-01-04,Egypt,Togo,2.0,1.0
1,2000-01-07,Tunisia,Togo,7.0,0.0
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0
4,2000-01-09,Guatemala,Armenia,1.0,1.0
5,2000-01-09,Ivory Coast,Egypt,2.0,0.0
6,2000-01-09,Mexico,Iran,2.0,1.0
7,2000-01-11,Bermuda,Canada,0.0,2.0
8,2000-01-11,Burkina Faso,Cameroon,2.0,2.0
9,2000-01-13,Senegal,Cameroon,0.0,0.0


In [73]:
matches_long["result"] = np.where(
    matches_long["goals_for"] > matches_long["goals_against"],
    3,
    np.where(
        matches_long["goals_for"] == matches_long["goals_against"],
        1,
        0
    )
)

In [74]:
matches_long[
    [
        "team",
        "goals_for",
        "goals_against",
        "result"
    ]
].head(20)

,team,goals_for,goals_against,result
0,Egypt,2.0,1.0,3
1,Tunisia,7.0,0.0,3
2,Trinidad and Tobago,0.0,0.0,1
3,Burkina Faso,1.0,1.0,1
4,Guatemala,1.0,1.0,1
5,Ivory Coast,2.0,0.0,3
6,Mexico,2.0,1.0,3
7,Bermuda,0.0,2.0,0
8,Burkina Faso,2.0,2.0,1
9,Senegal,0.0,0.0,1


## 12. Forma reciente de las selecciones

Se calculan estadísticas acumuladas de los últimos cinco partidos disputados por cada selección antes de cada encuentro.

In [75]:
# Ultimos 5 resultados

matches_long["last5_points"] = (
    matches_long
    .groupby("team")["result"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .sum()
    )
)

In [76]:
# Goles marcados 5 ultimos partidos

matches_long["last5_goals_for"] = (
    matches_long
    .groupby("team")["goals_for"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [77]:
# Goles encajados 5 ultimos partidos

matches_long["last5_goals_against"] = (
    matches_long
    .groupby("team")["goals_against"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [78]:
# Diferencia de goles

matches_long["goal_balance"] = (
    matches_long["goals_for"]
    - matches_long["goals_against"]
)

matches_long["last5_goal_balance"] = (
    matches_long
    .groupby("team")["goal_balance"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(5, min_periods=1)
         .mean()
    )
)

In [79]:
matches_long[
    [
        "date",
        "team",
        "result",
        "last5_points",
        "last5_goals_for",
        "last5_goals_against",
        "last5_goal_balance"
    ]
].head(30)

,date,team,result,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,2000-01-04,Egypt,3,NaN,NaN,NaN,NaN
1,2000-01-07,Tunisia,3,NaN,NaN,NaN,NaN
2,2000-01-08,Trinidad and Tobago,1,NaN,NaN,NaN,NaN
3,2000-01-09,Burkina Faso,1,NaN,NaN,NaN,NaN
4,2000-01-09,Guatemala,1,NaN,NaN,NaN,NaN
5,2000-01-09,Ivory Coast,3,NaN,NaN,NaN,NaN
6,2000-01-09,Mexico,3,NaN,NaN,NaN,NaN
7,2000-01-11,Bermuda,0,NaN,NaN,NaN,NaN
8,2000-01-11,Burkina Faso,1,1.0,1.0,1.0,0.0
9,2000-01-13,Senegal,1,NaN,NaN,NaN,NaN


In [80]:
#Recalcular last5_points

matches_long["last5_points"] = (
    matches_long
    .groupby("team")["result"]
    .transform(
        lambda x: x.shift().rolling(5, min_periods=1).sum()
    )
)

In [81]:
#Recalcular last5_goals_for

matches_long["last5_goals_for"] = (
    matches_long
    .groupby("team")["goals_for"]
    .transform(
        lambda x: x.shift().rolling(5, min_periods=1).sum()
    )
)

In [82]:
#Recalcular last5_goals_against

matches_long["last5_goals_against"] = (
    matches_long
    .groupby("team")["goals_against"]
    .transform(
        lambda x: x.shift().rolling(5, min_periods=1).sum()
    )
)

In [83]:
#Recalcular last5_goal_balance

matches_long["last5_goal_balance"] = (
    matches_long
    .groupby("team")["goal_balance"]
    .transform(
        lambda x: x.shift().rolling(5, min_periods=1).sum()
    )
)

In [84]:
#Comprobación

matches_long[
    [
        "date",
        "team",
        "result",
        "last5_points",
        "last5_goals_for",
        "last5_goals_against",
        "last5_goal_balance"
    ]
].head(30)

,date,team,result,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
0,2000-01-04,Egypt,3,NaN,NaN,NaN,NaN
1,2000-01-07,Tunisia,3,NaN,NaN,NaN,NaN
2,2000-01-08,Trinidad and Tobago,1,NaN,NaN,NaN,NaN
3,2000-01-09,Burkina Faso,1,NaN,NaN,NaN,NaN
4,2000-01-09,Guatemala,1,NaN,NaN,NaN,NaN
5,2000-01-09,Ivory Coast,3,NaN,NaN,NaN,NaN
6,2000-01-09,Mexico,3,NaN,NaN,NaN,NaN
7,2000-01-11,Bermuda,0,NaN,NaN,NaN,NaN
8,2000-01-11,Burkina Faso,1,1.0,1.0,1.0,0.0
9,2000-01-13,Senegal,1,NaN,NaN,NaN,NaN


In [85]:
matches_long[
    [
        "last5_points",
        "last5_goals_for",
        "last5_goals_against",
        "last5_goal_balance"
    ]
].describe()

,last5_points,last5_goals_for,last5_goals_against,last5_goal_balance
count,45751.000000,45751.000000,45751.000000,45751.000000
mean,6.868724,6.637407,6.594785,0.042622
std,3.719767,3.970795,4.357998,6.800622
min,0.000000,0.000000,0.000000,-67.000000
25%,4.000000,4.000000,4.000000,-4.000000
50%,7.000000,6.000000,6.000000,0.000000
75%,10.000000,9.000000,8.000000,4.000000
max,15.000000,68.000000,67.000000,67.000000


In [86]:
matches_long[
    [
        "last5_points",
        "last5_goal_balance"
    ]
].corr()

,last5_points,last5_goal_balance
last5_points,1.000000,0.845805
last5_goal_balance,0.845805,1.000000


In [89]:
form_home = matches_long[
    [
        "date",
        "team",
        "last5_points",
        "last5_goals_for",
        "last5_goals_against",
        "last5_goal_balance"
    ]
].copy()

form_away = form_home.copy()

In [90]:
form_home = form_home.rename(
    columns={
        "team": "home_team",
        "last5_points": "home_last5_points",
        "last5_goals_for": "home_last5_goals_for",
        "last5_goals_against": "home_last5_goals_against",
        "last5_goal_balance": "home_last5_goal_balance"
    }
)

form_away = form_away.rename(
    columns={
        "team": "away_team",
        "last5_points": "away_last5_points",
        "last5_goals_for": "away_last5_goals_for",
        "last5_goals_against": "away_last5_goals_against",
        "last5_goal_balance": "away_last5_goal_balance"
    }
)

In [91]:
form_home.head()

,date,home_team,home_last5_points,home_last5_goals_for,home_last5_goals_against,home_last5_goal_balance
0,2000-01-04,Egypt,NaN,NaN,NaN,NaN
1,2000-01-07,Tunisia,NaN,NaN,NaN,NaN
2,2000-01-08,Trinidad and Tobago,NaN,NaN,NaN,NaN
3,2000-01-09,Burkina Faso,NaN,NaN,NaN,NaN
4,2000-01-09,Guatemala,NaN,NaN,NaN,NaN
